In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')


df = pd.read_csv("/content/drive/MyDrive/Clean Data(Sheet2).csv")




Mounted at /content/drive


# Contextual Relevancy Metric

In [ ]:

import os
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "50000"
os.environ["DEEPEVAL_TASK_GATHER_BUFFER_SECONDS_OVERRIDE"] = "200"
import asyncio
from ollama import AsyncClient, Client
from deepeval.models import DeepEvalBaseLLM
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import GEval
# OS New Edits
from deepeval.metrics import (
  ContextualRelevancyMetric,
  ContextualRecallMetric,
  ContextualPrecisionMetric,
  AnswerRelevancyMetric,
  FaithfulnessMetric
)



class CustomOllamaLlama(DeepEvalBaseLLM):
    def __init__(self, model_name: str = "llama3.1:8b"):
    # def __init__(self, model_name: str = "mistral"):


        self.model_name = model_name
        self.client = Client()
        self.async_client = AsyncClient()

    def load_model(self):
        return self.model_name

    def generate(self, prompt: str) -> str:
        response = self.client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},


        )
        return response["message"]["content"]

    async def a_generate(self, prompt: str) -> str:
        response = await self.async_client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},
        )
        return response["message"]["content"]

    def get_model_name(self) -> str:
        return f"Ollama ({self.model_name})"


custom_llm = CustomOllamaLlama(model_name="llama3.1:8b")
# custom_llm = CustomOllamaLlama(model_name="mistral")


contextual_relevancy_metric = GEval(
    name="Contextual Relevancy",
    evaluation_steps=[
        "This metric evaluates retrieval quality only — it does not involve the 'actual output' at all.",
        "The 'context' is retrieved guideline knowledge being assessed on its own merit, not on how it will later be phrased or used.",
        "Identify what the 'input' task prompt is asking for.",
        "Assess what proportion of the 'context' (retrieved guideline material) is genuinely relevant and useful for answering that specific input, versus off-topic, overly generic, or unrelated guideline content.",
        "Base the score on the proportion of the context that is relevant, not solely on whether any irrelevant content is severe enough to notice."
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.CONTEXT],
    threshold=0.5,
    model=custom_llm
)



# Load test cases from the csv
df = pd.read_csv("/content/drive/MyDrive/Clean Data(Sheet2).csv")


test_cases = [
    LLMTestCase(
        input=row["Input"],
        actual_output=row["Actual_output"],
        context=[str(row["Expected_output"])]
    )
    for _, row in df.iterrows()
]
print(f"Loaded {len(test_cases)} test cases")

evaluate(
    test_cases,
    [contextual_relevancy_metric]
    # , # run 5 at  a time instead of all at once
       , async_config=AsyncConfig(
        run_async=True,
        max_concurrent=5,      # lower this since each task is slow on your setup
        throttle_value=1       # 1 second delay between task submissions
    )
)

Loaded 72 test cases


✨ You're running DeepEval's latest Contextual Relevancy [GEval] Metric! (using Ollama (llama3.1:8b), strict=False,
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_4 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_5 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_6 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_7 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_8 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_9 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                  

⚠ WARNING: No hyperparameters logged.
» ]8;id=934833;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 305.63s | token cost: None)
» Test Results (111 total tests):
   » Pass Rate: 86.49% | Passed: 96 | Failed: 15

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Contextual Relevancy [GEval]', threshold=0.5, success=True, score=0.8, reason='The context is mostly relevant for answering the input question, providing specific information about who the guide is intended for. However, some parts of the context are overly generic or provide unnecessary details.', strict_mode=False, evaluation_model='Ollama (llama3.1:8b)', error=None, evaluation_cost=None, input_tokens=None, output_tokens=None, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "This metric evaluates retrieval quality only — it does not involve the \'actual output\' at all.",\n    "The \'context\' is retrieved guideline knowledge being assessed on its own merit, not on how it will later be phrased or used.",\n    "Identify what the \'input\' task prompt is asking for.",\n    "Assess what proportion of the \'context\' (retrieved guideline material) is genuinely relevant 

# Contextual Precision Metric

In [ ]:

import os
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "50000"
os.environ["DEEPEVAL_TASK_GATHER_BUFFER_SECONDS_OVERRIDE"] = "200"
import asyncio
from ollama import AsyncClient, Client
from deepeval.models import DeepEvalBaseLLM
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import GEval
# OS New Edits
from deepeval.metrics import (
  ContextualRelevancyMetric,
  ContextualRecallMetric,
  ContextualPrecisionMetric,
  AnswerRelevancyMetric,
  FaithfulnessMetric
)



class CustomOllamaLlama(DeepEvalBaseLLM):
    def __init__(self, model_name: str = "llama3.1:8b"):
    # def __init__(self, model_name: str = "mistral"):


        self.model_name = model_name
        self.client = Client()
        self.async_client = AsyncClient()

    def load_model(self):
        return self.model_name

    def generate(self, prompt: str) -> str:
        response = self.client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},


        )
        return response["message"]["content"]

    async def a_generate(self, prompt: str) -> str:
        response = await self.async_client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},
        )
        return response["message"]["content"]

    def get_model_name(self) -> str:
        return f"Ollama ({self.model_name})"


custom_llm = CustomOllamaLlama(model_name="llama3.1:8b")
# custom_llm = CustomOllamaLlama(model_name="mistral")

contextual_precision_metric = GEval(
    name="Contextual Precision",
    evaluation_steps=[
        "This metric evaluates scope only — whether the 'actual output' stays within the bounds of what the 'context' covers, not whether its content is factually accurate.",
        "The 'context' is retrieved guideline knowledge, not a literal target — rewording, restructuring, or synthesizing its content in the actual output's own style is not by itself scope drift.",
        "Identify any content in the 'actual output' that introduces topics, rules, or guidance not addressed by the context at all — as opposed to content that merely expresses the same guidance differently.",
        "Do not factor in factual inaccuracy or contradiction — that is evaluated separately by the Correctness and Faithfulness metrics, not this one.",
        "Do not factor in omitted content from the context — that is evaluated separately by the Contextual Recall metric, not this one.",
        "Assess what proportion of the actual output stays tightly scoped to the context's subject matter, versus unrelated additions, unnecessary elaboration, or drift into topics it does not address.",
        "Base the score on the proportion of the actual output that drifts beyond scope, not solely on whether any drift is severe enough to notice."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT],
    threshold=0.5,
    model=custom_llm
)


# Load test cases from the csv
df = pd.read_csv("/content/drive/MyDrive/Clean Data(Sheet2).csv")


test_cases = [
    LLMTestCase(
        input=row["Input"],
        actual_output=row["Actual_output"],
        context=[str(row["Expected_output"])]
    )
    for _, row in df.iterrows()
]
print(f"Loaded {len(test_cases)} test cases")

evaluate(
    test_cases,
    [contextual_precision_metric]
    # , # run 5 at  a time instead of all at once
       , async_config=AsyncConfig(
        run_async=True,
        max_concurrent=5,      # lower this since each task is slow on your setup
        throttle_value=1       # 1 second delay between task submissions
    )
)

Loaded 72 test cases


✨ You're running DeepEval's latest Contextual Precision [GEval] Metric! (using Ollama (llama3.1:8b), strict=False,
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_4 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_5 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_6 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_7 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_8 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_9 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_10 (Passed 1 metrics)                                

⚠ WARNING: No hyperparameters logged.
» ]8;id=7086;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 198.9s | token cost: None)
» Test Results (72 total tests):
   » Pass Rate: 95.83% | Passed: 69 | Failed: 3

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Contextual Precision [GEval]', threshold=0.5, success=True, score=0.9, reason="The actual output closely mirrors the context, with a minor addition of unnecessary elaboration on who is included in 'any team member'. The vast majority of the content remains tightly scoped to the context's subject matter.", strict_mode=False, evaluation_model='Ollama (llama3.1:8b)', error=None, evaluation_cost=None, input_tokens=None, output_tokens=None, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "This metric evaluates scope only — whether the \'actual output\' stays within the bounds of what the \'context\' covers, not whether its content is factually accurate.",\n    "The \'context\' is retrieved guideline knowledge, not a literal target — rewording, restructuring, or synthesizing its content in the actual output\'s own style is not by itself scope drift.",\n    "Identify any con

# Contextual Recall Metric

In [ ]:

import os
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "50000"
os.environ["DEEPEVAL_TASK_GATHER_BUFFER_SECONDS_OVERRIDE"] = "200"
import asyncio
from ollama import AsyncClient, Client
from deepeval.models import DeepEvalBaseLLM
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import GEval
# OS New Edits
from deepeval.metrics import (
  ContextualRelevancyMetric,
  ContextualRecallMetric,
  ContextualPrecisionMetric,
  AnswerRelevancyMetric,
  FaithfulnessMetric
)



class CustomOllamaLlama(DeepEvalBaseLLM):
    def __init__(self, model_name: str = "llama3.1:8b"):
    # def __init__(self, model_name: str = "mistral"):


        self.model_name = model_name
        self.client = Client()
        self.async_client = AsyncClient()

    def load_model(self):
        return self.model_name

    def generate(self, prompt: str) -> str:
        response = self.client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},


        )
        return response["message"]["content"]

    async def a_generate(self, prompt: str) -> str:
        response = await self.async_client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},
        )
        return response["message"]["content"]

    def get_model_name(self) -> str:
        return f"Ollama ({self.model_name})"


custom_llm = CustomOllamaLlama(model_name="llama3.1:8b")
# custom_llm = CustomOllamaLlama(model_name="mistral")

contextual_recall_metric = GEval(
    name="Contextual Recall",
    evaluation_steps=[
        "This metric evaluates coverage only — whether the 'actual output' includes all guidance points from the 'context' that are relevant to the 'input', not whether that content is stated accurately.",
        "The 'context' is retrieved guideline knowledge, not a literal target — a point counts as covered if its substance is present in the actual output, even when phrased very differently, condensed, or restructured.",
        "Identify the guidance points, rules, or details in the 'context' that are relevant to answering the 'input' task prompt.",
        "For each point, assess whether its substance is fully present, partially present, or absent in the 'actual output', regardless of wording similarity to the context.",
        "Do not factor in factual inaccuracy in what is included — that is evaluated separately by the Correctness metric, not this one.",
        "Do not factor in additional content beyond the context's scope — that is evaluated separately by the Contextual Precision metric, not this one.",
        "Base the score on the proportion of relevant points whose substance is covered, not solely on whether any single omission is severe enough to notice."
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT],
    threshold=0.5,
    model=custom_llm
)


# Load test cases from the csv
df = pd.read_csv("/content/drive/MyDrive/Clean Data(Sheet2).csv")


test_cases = [
    LLMTestCase(
        input=row["Input"],
        actual_output=row["Actual_output"],
        context=[str(row["Expected_output"])]
    )
    for _, row in df.iterrows()
]
print(f"Loaded {len(test_cases)} test cases")

evaluate(
    test_cases,
    [contextual_recall_metric ]
    # , # run 5 at  a time instead of all at once
       , async_config=AsyncConfig(
        run_async=True,
        max_concurrent=5,      # lower this since each task is slow on your setup
        throttle_value=1       # 1 second delay between task submissions
    )
)

Loaded 72 test cases


✨ You're running DeepEval's latest Contextual Recall [GEval] Metric! (using Ollama (llama3.1:8b), strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_4 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_5 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_6 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_7 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_8 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_9 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_10 (Passed 1 metrics)                                

⚠ WARNING: No hyperparameters logged.
» ]8;id=244280;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 229.31s | token cost: None)
» Test Results (72 total tests):
   » Pass Rate: 98.61% | Passed: 71 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Contextual Recall [GEval]', threshold=0.5, success=True, score=0.9, reason='The actual output fully includes the substance of all relevant guidance points from the context, with some minor differences in wording. The list of target audiences is identical, including specific roles and tasks.', strict_mode=False, evaluation_model='Ollama (llama3.1:8b)', error=None, evaluation_cost=None, input_tokens=None, output_tokens=None, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "This metric evaluates coverage only — whether the \'actual output\' includes all guidance points from the \'context\' that are relevant to the \'input\', not whether that content is stated accurately.",\n    "The \'context\' is retrieved guideline knowledge, not a literal target — a point counts as covered if its substance is present in the actual output, even when phrased very differently, condensed,